# 8.2 只改提示，回答為什麼可能改變？


同樣問「2+2=?」，一次要求「用一句話回答」，另一次要求「用貼切比喻回答」。第一種希望直接交代4；第二種還希望有分組與合併的例子。題目與模型都可以保持相同，改變的只是這次送進去的文字。

這份提示文字叫**prompt**。模型看提示與已寫出的回答，逐步預測下一個文字片段；要求也在它讀到的前文裡，因而可能改變下一步的選擇。模型內部可調的數字叫**權重**，這次只換提示，沒有更新權重，也沒有把要求永久教進模型。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
question = "2+2=?"
requests = ["用一句話回答", "用貼切比喻回答"]
for style in requests:
    prompt = style + "：" + question
    print(prompt)
print("兩次都使用同一份權重與同一個選字方式")

用一句話回答：2+2=?
用貼切比喻回答：2+2=?
兩次都使用同一份權重與同一個選字方式


程式印出兩份待比較的提示，不會呼叫模型。第一行是`用一句話回答：2+2=?`，第二行是`用貼切比喻回答：2+2=?`。要實際觀察提示效果，才把這兩份文字交給同一模型，按8.1的數字與比喻判準分別記錄。

比較時也固定選字方式。例如每一步取最高分候選，叫貪婪選擇（greedy）；如果每次隨機抽字，一次不同回答也可能來自抽樣。模型若還不會算加法或不懂比喻，換提示未必能補出能力。新對話若沒保留這個要求，也不能假定仍有同一寫法。

練習只把第二個要求改成「只回一個數字」。執行前先預測：第一份提示完全不變，第二份只有要求改變。這是我們希望在後面的真實比較中保持的條件。

<details>
<summary>補充：既有實驗的條件與完整紀錄</summary>

後面的實跑也保留了真正「只換提示」的比較：先把同一模型訓練成可讀`style=concise`、`style=vivid`與`style=json`，評估時固定權重與最高分選字，只換同一道加法題前的style值。[8.4](https://birdhackor.github.io/tiny-perceptron-vlm/8.4.html)列出它的三種實際回答。這支持的是已教過這三種條件後的切換，不是隨便加一種新風格名字，模型就自動懂。

</details>
